|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Admission and preemption<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the scheduler that does not fall over<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the scheduler that does not fall over.

Stage 05's scheduler assumed memory was infinite. This one has a block budget
that a running sequence can exhaust at any step, and it has to stay correct
when that happens.

All simulation. The GPU is a counter, which is the right way to get this
right before it is fast.

In [ ]:
### run this cell

BLOCK   = 16
PROMPT  = 48
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=5000).astype(int) + 1

from dataclasses import dataclass

@dataclass(eq=False)
class Sequence:
  length: int        # tokens that it holds now: the prompt, then each new token
  target: int        # the length at which it finishes

def blocks_for(num_tokens):
  return -(-num_tokens // BLOCK)                # ceiling division

print(f'median sequence: {blocks_for(int(np.median(lengths)))} blocks, p99: {blocks_for(int(np.percentile(lengths,99)))} blocks')

# Exercise 1: three queues and a budget

Four methods: `arrive`, `admit`, `preempt`, `step`.

The interesting one is `step`. A sequence crossing into a new block may find
the pool empty, and the answer is not to fail. It is to take the blocks back
from somebody else.

In [ ]:
class Scheduler:
  def __init__(self, pool_blocks, watermark=0.05, max_running=64):
    self.free = pool_blocks
    # never admit a request that would leave less than this free
    self.reserve = max(1, int(watermark*pool_blocks))
    self.max_running = max_running          # max_num_seqs
    self.waiting, self.running = [], []
    self.preemptions = self.recomputed = self.work = 0

  def arrive(self, total_len):
    self.waiting.append(Sequence(PROMPT, total_len))

  def admit(self):
    # move requests from waiting to running while the reserve holds
    # and there is a free slot (max_running)
    

  def preempt(self, protect):
    # take the NEWEST running sequence's blocks and send it back to the
    # front of the queue. Never preempt `protect`, it is mid-step.
    

  def step(self):
    self.admit()
    for seq in list(self.running):
      if seq not in self.running:
        continue
      # does this token start a new block?
      if seq.length % BLOCK == 0:
        # no room: preempt until there is, then take a block
        
      seq.length += 1
      # finished? give everything back
      

scheduler = Scheduler(2000)
for index in range(400):
  scheduler.arrive(PROMPT + int(lengths[index]))
steps = 0
while (scheduler.waiting or scheduler.running) and steps < 200000:
  scheduler.step()
  steps += 1
print(f'{steps:,} steps, {scheduler.preemptions} preemptions, '
      f'{100*scheduler.recomputed/scheduler.work:.1f}% of the work was done twice')

# Exercise 2: how hard can you squeeze?

Sweep the pool size. Watch for where degradation stops being graceful.

In [ ]:
def run(pool, watermark=0.05, num_requests=400):
  scheduler = Scheduler(pool, watermark)
  for index in range(num_requests):
    scheduler.arrive(PROMPT + int(lengths[index]))
  steps = 0
  while (scheduler.waiting or scheduler.running) and steps < 200000:
    scheduler.step()
    steps += 1
  return steps, scheduler.preemptions, scheduler.recomputed, scheduler.work

pools = [300, 500, 750, 1000, 1500, 2000, 3000, 4000]
rows = 
roomy_steps = rows[-1][0]

print(f"{'pool':>6} {'steps':>8} {'vs roomy':>9} {'preempt':>8} {'wasted work':>12}")
for pool, (steps, preemptions, recomputed, work) in zip(pools, rows):
  print(f'{pool:>6} {steps:>8,} {steps/roomy_steps:>8.2f}x {preemptions:>8} '
        f'{100*recomputed/work:>11.1f}%')

# Exercise 3: the watermark

The reserve is the only thing stopping a new arrival from starving everyone
already running. Try turning it off.

In [ ]:
# the watermark is the only knob that decides how greedy admission is.
# 0% admits until the pool is empty. 50% keeps half the pool in reserve.
print(f"{'watermark':>10} {'steps':>8} {'preempt':>8} {'recomputed':>11}")
for watermark in (0.0, 0.01, 0.05, 0.15, 0.30, 0.50):
  steps, preemptions, recomputed, work = 
  print(f'{watermark:>10.0%} {steps:>8,} {preemptions:>8} {100*recomputed/work:>11.1f}%')

### Before you open the solution

1. Find where the steps column turns up as the pool becomes smaller. How far
   can you squeeze before the graceful decrease becomes a collapse?
2. A watermark of 0% gives the most preemptions and the most repeated work. It
   is still near the fastest on total steps. How can both facts be true? Think
   about what a preempted sequence gives back.
3. At 50% there are no preemptions, and the step count is the worst on the
   table. What does the machine do with the other half of its pool?
4. You preempt `running[-1]`, the newest sequence. What changes if you preempt
   `running[0]`? Count the tokens that each one must make again.